In [1]:
%cd /root/vs-gats-hhi
import os
import h5py
import numpy as np
from tqdm import tqdm

import utils.io as io
from datasets.hico_constants import HicoConstants
from utils.bbox_utils import compute_area
# from datasets.metadata import hico_classes

data_const = HicoConstants()

/root/vs-gats-hhi
/root/vs-gats-hhi/datasets/hico


In [2]:
equipment_idx = [14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 72, 74]
material_idx = [67, 68, 69, 70, 71, 73, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160]
human_idx = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


# Mapping class
wico_hhi_classes = [
    '__background__', # Always index 0 # 0
    'human_engineer',    #-- 1/6--> human # 1
    'human_helper',    #-- 2/7 --> human # 2
    'human_master',    #-- 3/8 --> human # 3
    'human_operator',    #-- 4 --> human # 4
    'human_safety_officer',    #-- 5 --> human # 5
    'human_supervisor',    #-- 11 --> human # 6
    'human_welder',    #-- 12/9 --> human # 7
    'human_worker',    #-- 13/10 --> human # 8
    'material', # 9
    'equipment' # 10
]

# mapping (keys and values as integers)
index_mapping = {
    **{index: 10 for index in equipment_idx},  # Map all Equipment indices to 10
    **{index: 9 for index in material_idx},    # Map all Material indices to 9
    # Add other mappings if needed
    6: 1,
    7: 2,
    8: 3,
    11:6,
    12:7,
    9: 7,
    10: 8,
    13:8,
}

# Mapping function
def map_value(value):
    return index_mapping.get(value, value) 


fasterrcnn_classes = [
    '__background__',    #-- 0 --> 
    'human_engineer',    #-- 1 --> human
    'human_helper',    #-- 2 --> human
    'human_master',    #-- 3 --> human
    'human_operator',    #-- 4 --> human
    'human_safety_officer',    #-- 5 --> human
    'human_squating_engineer',    #-- 6 --> human
    'human_squating_helper',    #-- 7 --> human
    'human_squating_master',    #-- 8 --> human
    'human_squating_welder',    #-- 9 --> human
    'human_squating_worker',    #-- 10 --> human
    'human_supervisor',    #-- 11 --> human
    'human_welder',    #-- 12 --> human
    'human_worker',    #-- 13 --> human
    'equipment_bucket',    #-- 14 --> equipment
    'equipment_carried_concrete_leveler',    #-- 15 --> equipment
    'equipment_carried_drawing',    #-- 16 --> equipment
    'equipment_carried_general_tools',    #-- 17 --> equipment
    'equipment_carried_hammer',    #-- 18 --> equipment
    'equipment_carried_screwdriver',    #-- 19 --> equipment
    'equipment_carried_wire_cutter',    #-- 20 --> equipment
    'equipment_scaffolding',    #-- 21 --> equipment
    'equipment_unused_bag',    #-- 22 --> equipment
    'equipment_unused_bucket',    #-- 23 --> equipment
    'equipment_unused_cable_roller',    #-- 24 --> equipment
    'equipment_unused_car',    #-- 25 --> equipment
    'equipment_unused_cart',    #-- 26 --> equipment
    'equipment_unused_chair',    #-- 27 --> equipment
    'equipment_unused_cleaning_tools',    #-- 28 --> equipment
    'equipment_unused_concrete_finisher',    #-- 29 --> equipment
    'equipment_unused_concrete_leveler',    #-- 30 --> equipment
    'equipment_unused_cutting_machine',    #-- 31 --> equipment
    'equipment_unused_cutting_table',    #-- 32 --> equipment
    'equipment_unused_drawing',    #-- 33 --> equipment
    'equipment_unused_excavator',    #-- 34 --> equipment
    'equipment_unused_fan',    #-- 35 --> equipment
    'equipment_unused_gas_tank',    #-- 36 --> equipment
    'equipment_unused_general_tools',    #-- 37 --> equipment
    'equipment_unused_hammer',    #-- 38 --> equipment
    'equipment_unused_hose',    #-- 39 --> equipment
    'equipment_unused_ladder',    #-- 40 --> equipment
    'equipment_unused_measuring_tape',    #-- 41 --> equipment
    'equipment_unused_mobile_crane',    #-- 42 --> equipment
    'equipment_unused_rebar_machine',    #-- 43 --> equipment
    'equipment_unused_rubber_cone',    #-- 44 --> equipment
    'equipment_unused_scaffolding',    #-- 45 --> equipment
    'equipment_unused_theodolite',    #-- 46 --> equipment
    'equipment_unused_truck',    #-- 47 --> equipment
    'equipment_unused_welding_machine',    #-- 48 --> equipment
    'equipment_used_bucket',    #-- 49 --> equipment
    'equipment_used_cart',    #-- 50 --> equipment
    'equipment_used_cleaning_tools',    #-- 51 --> equipment
    'equipment_used_concrete_finisher',    #-- 52 --> equipment
    'equipment_used_concrete_leveler',    #-- 53 --> equipment
    'equipment_used_concrete_pump',    #-- 54 --> equipment
    'equipment_used_concrete_vibrator',    #-- 55 --> equipment
    'equipment_used_cutting_table',    #-- 56 --> equipment
    'equipment_used_drawing',    #-- 57 --> equipment
    'equipment_used_general_tools',    #-- 58 --> equipment
    'equipment_used_hammer',    #-- 59 --> equipment
    'equipment_used_ladder',    #-- 60 --> equipment
    'equipment_used_measuring_tape',    #-- 61 --> equipment
    'equipment_used_mobile_crane',    #-- 62 --> equipment
    'equipment_used_nail',    #-- 63 --> equipment
    'equipment_used_rebar_machine',    #-- 64 --> equipment
    'equipment_used_theodolite',    #-- 65 --> equipment
    'equipment_used_welding_machine',    #-- 66 --> equipment
    'material_column_rebar_global_25',    #-- 67 --> material
    'material_column_rebar_local_25',    #-- 68 --> material
    'material_column_rebar_local_50',    #-- 69 --> material
    'material_concrete_global_100',    #-- 70 --> material
    'material_concrete_global_25',    #-- 71 --> material
    'equipment_used_nailing_machine',    #-- 72 --> equipment
    'material_concrete_global_50',    #-- 73 --> material
    'equipment_used_scaffolding',    #-- 74 --> equipment
    'material_concrete_global_75',    #-- 75 --> material
    'material_concrete_local_100',    #-- 76 --> material
    'material_concrete_local_50',    #-- 77 --> material
    'material_concrete_local_75',    #-- 78 --> material
    'material_finished_plywood_formwork_global_100',    #-- 79 --> material
    'material_finished_plywood_formwork_global_25',    #-- 80 --> material
    'material_finished_plywood_formwork_global_75',    #-- 81 --> material
    'material_finished_plywood_formwork_local_75',    #-- 82 --> material
    'material_finished_steel_formwork_global_25',    #-- 83 --> material
    'material_finished_steel_formwork_global_75',    #-- 84 --> material
    'material_finished_steel_formwork_local_100',    #-- 85 --> material
    'material_finished_steel_formwork_local_25',    #-- 86 --> material
    'material_rebar_rod_local_0',    #-- 87 --> material
    'material_slab_rebar_global_100',    #-- 88 --> material
    'material_slab_rebar_global_25',    #-- 89 --> material
    'material_slab_rebar_global_50',    #-- 90 --> material
    'material_slab_rebar_global_75',    #-- 91 --> material
    'material_slab_rebar_local_100',    #-- 92 --> material
    'material_slab_rebar_local_25',    #-- 93 --> material
    'material_slab_rebar_local_50',    #-- 94 --> material
    'material_slab_rebar_local_75',    #-- 95 --> material
    'material_unfinished_plywood_formwork_local_0',    #-- 96 --> material
    'material_unfinished_styrofoam_formwork_local_0',    #-- 97 --> material
    'material_unused_column_rebar_local_25',    #-- 98 --> material
    'material_unused_column_rebar_local_25-',    #-- 99 --> material
    'material_unused_column_rebar_local_50',    #-- 100 --> material
    'material_unused_concrete_local_100',    #-- 101 --> material
    'material_unused_concrete_local_25',    #-- 102 --> material
    'material_unused_concrete_local_50',    #-- 103 --> material
    'material_unused_finished_plywood_formwork_local_50',    #-- 104 --> material
    'material_unused_finished_plywood_formwork_local_75',    #-- 105 --> material
    'material_unused_finished_steel_formwork_local_75',    #-- 106 --> material
    'material_unused_plywood_formwork_local_0',    #-- 107 --> material
    'material_unused_plywood_formwork_local_75',    #-- 108 --> material
    'material_unused_rebar_column_local_25',    #-- 109 --> material
    'material_unused_rebar_rod_local_0',    #-- 110 --> material
    'material_unused_slab_rebar_global_0',    #-- 111 --> material
    'material_unused_slab_rebar_global_100',    #-- 112 --> material
    'material_unused_slab_rebar_local_100',    #-- 113 --> material
    'material_unused_slab_rebar_local_25',    #-- 114 --> material
    'material_unused_slab_rebar_local_75',    #-- 115 --> material
    'material_unused_unfinished_plywood_formwork_local_0',    #-- 116 --> material
    'material_used_column_rebar_global_25',    #-- 117 --> material
    'material_used_column_rebar_local_25',    #-- 118 --> material
    'material_used_column_rebar_local_50',    #-- 119 --> material
    'material_used_column_rebar_local_75',    #-- 120 --> material
    'material_used_concrete_global_0',    #-- 121 --> material
    'material_used_concrete_global_100',    #-- 122 --> material
    'material_used_concrete_global_25',    #-- 123 --> material
    'material_used_concrete_global_50',    #-- 124 --> material
    'material_used_concrete_local_0',    #-- 125 --> material
    'material_used_concrete_local_100',    #-- 126 --> material
    'material_used_concrete_local_25',    #-- 127 --> material
    'material_used_concrete_local_50',    #-- 128 --> material
    'material_used_concrete_local_75',    #-- 129 --> material
    'material_used_finished_plywood_formwork_global_100',    #-- 130 --> material
    'material_used_finished_plywood_formwork_global_25',    #-- 131 --> material
    'material_used_finished_plywood_formwork_global_50',    #-- 132 --> material
    'material_used_finished_plywood_formwork_global_75',    #-- 133 --> material
    'material_used_finished_plywood_formwork_local_100',    #-- 134 --> material
    'material_used_finished_plywood_formwork_local_25',    #-- 135 --> material
    'material_used_finished_plywood_formwork_local_50',    #-- 136 --> material
    'material_used_finished_plywood_formwork_local_75',    #-- 137 --> material
    'material_used_finished_steel_formwork_global_25',    #-- 138 --> material
    'material_used_finished_steel_formwork_global_75',    #-- 139 --> material
    'material_used_finished_steel_formwork_local_100',    #-- 140 --> material
    'material_used_finished_steel_formwork_local_50',    #-- 141 --> material
    'material_used_finished_steel_formwork_local_75',    #-- 142 --> material
    'material_used_plywood_formwork_global_100',    #-- 143 --> material
    'material_used_plywood_formwork_global_25',    #-- 144 --> material
    'material_used_rebar_rod_global_0',    #-- 145 --> material
    'material_used_rebar_rod_local_0',    #-- 146 --> material
    'material_used_slab_rebar_global_0',    #-- 147 --> material
    'material_used_slab_rebar_global_100',    #-- 148 --> material
    'material_used_slab_rebar_global_25',    #-- 149 --> material
    'material_used_slab_rebar_global_50',    #-- 150 --> material
    'material_used_slab_rebar_local_100',    #-- 151 --> material
    'material_used_slab_rebar_local_25',    #-- 152 --> material
    'material_used_slab_rebar_local_50',    #-- 153 --> material
    'material_used_slab_rebar_local_75',    #-- 154 --> material
    'material_used_steel_formwork_global_100',    #-- 155 --> material
    'material_used_unfinished_plywood_formwork_global_0',    #-- 156 --> material
    'material_used_unfinished_plywood_formwork_local_0',    #-- 157 --> material
    'material_used_unfinished_plywood_formwork_local_50',    #-- 158 --> material
    'material_used_unfinished_steel_formwork_local_0',    #-- 159 --> material
    'material_used_unfinished_styrofoam_formwork_local_0',    #-- 160 --> material
]



In [3]:
human = []
equipment = []
material = []

for idx,cls_name in enumerate(fasterrcnn_classes):
    header = cls_name.split('_')[0]
    if header == 'equipment':
        equipment.append(idx)
    elif header == 'material':
        material.append(idx)
    elif header == 'human':
        human.append(idx)

print(f'Equipment {equipment}')
print(f'Material {material}')
print(f'Human {human}')

Equipment [14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 72, 74]
Material [67, 68, 69, 70, 71, 73, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160]
Human [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


In [4]:
def select_det_ids(boxes,scores,nms_keep_ids,score_thresh,max_dets, required=False):
    if nms_keep_ids is None:
        nms_keep_ids = np.arange(0,scores.shape[0])
    
    # Select non max suppressed dets
    nms_scores = scores[nms_keep_ids]
    nms_boxes = boxes[nms_keep_ids]

    # Select dets above a score_thresh and which have area > 1
    nms_ids_above_thresh = np.nonzero(nms_scores > score_thresh)[0]
    nms_ids = []
    for i in range(min(nms_ids_above_thresh.shape[0],max_dets)):
        area = compute_area(nms_boxes[i],invalid=-1)
        if area > 1:
            nms_ids.append(i)
        
    # If no dets satisfy previous criterion select the highest ranking one with area > 1
    if len(nms_ids)==0:
        if required:
            nms_ids.append(np.argmax(nms_scores))
        # for i in range(nms_keep_ids.shape[0]):
        #     area = compute_area(nms_boxes[i],invalid=-1)
        #     if area > 1:
        #         nms_ids = [i]
        #         break
        else:
            return []
    # Convert nms ids to box ids
    nms_ids = np.array(nms_ids,dtype=np.int32)
    try:
        ids = nms_keep_ids[nms_ids]
    except:
        import pdb; pdb.set_trace()

    return ids

In [5]:
def select_dets_custom(
    boxes,
    scores,
    nms_keep_indices,
):
    selected_dets = []
    
    start_end_ids = np.zeros([len(fasterrcnn_classes)-1, 2], dtype=np.int32)
    start_id = 0
    
    # Filter Treshold object
    for cls_ind, cls_name in enumerate(fasterrcnn_classes):
        if cls_ind == 0:
            # Remove the predictions with background label
            continue
        cls_boxes = boxes[:, 4*cls_ind:4*(cls_ind + 1)]
        cls_scores = scores[:, cls_ind]
        cls_nms_keep_ids = np.array(nms_keep_indices[cls_ind])

        # guarantee at least have one person detection
        
        if 'human' in cls_name:
            select_ids = select_det_ids(
                cls_boxes,
                cls_scores,
                cls_nms_keep_ids,
                data_const.human_score_thresh,
                data_const.max_num_human,
                required=True)
                
        elif cls_name == '__background__':
            select_ids = select_det_ids(
                cls_boxes,
                cls_scores,
                cls_nms_keep_ids,
                data_const.background_score_thresh,
                data_const.max_num_background)
        else:
            select_ids = select_det_ids(
                cls_boxes,
                cls_scores,
                cls_nms_keep_ids,
                data_const.object_score_thresh,
                data_const.max_num_objects_per_class)
            
        
        try:
            if len(select_ids)==0 :
                boxes_scores_rpn_id_label = np.empty((0,7))
        except:
            import ipdb; ipdb.set_trace()

        else:
            boxes_scores_rpn_id_label = np.concatenate((
                cls_boxes[select_ids],
                np.expand_dims(cls_scores[select_ids],1),
                np.expand_dims(select_ids,1),
                np.expand_dims([cls_ind] * len(select_ids),1)), 1)

        selected_dets.append(boxes_scores_rpn_id_label)
        num_boxes = boxes_scores_rpn_id_label.shape[0]
        start_end_ids[cls_ind-1,:] = [start_id,start_id+num_boxes]
        start_id += num_boxes

    selected_dets = np.concatenate(selected_dets)
    
    # Mapping to wico_hhi_classes 1 - 10
    
    start_end_ids = np.vectorize(map_value)(start_end_ids)
    
    selected_dets[:, -1] = [index_mapping.get(val, val) for val in selected_dets[:, -1]]
    
    
    # guarantee at least have one object selected det id class already mapping into number 0 - 10
    
    object_selected_dets = []
    if selected_dets.shape[0] == 1:
        for cls_ind, cls_name in enumerate(wico_hhi_classes):
            if cls_ind >= 0 and cls_ind <= 8:
                # remove the predictions with background label
                continue
            cls_boxes = boxes[:, 4*cls_ind:4*(cls_ind + 1)]
            cls_scores = scores[:, cls_ind]
            cls_nms_keep_ids = np.array(nms_keep_indices[cls_ind])

            select_ids = select_det_ids(
                cls_boxes,
                cls_scores,
                cls_nms_keep_ids,
                data_const.object_score_thresh,
                data_const.max_num_objects_per_class,
                required=True)

            if len(select_ids)==0 :
                boxes_scores_rpn_id_label = np.empty((0,7))
            else:
                boxes_scores_rpn_id_label = np.concatenate((
                    cls_boxes[select_ids],
                    np.expand_dims(cls_scores[select_ids],1),
                    np.expand_dims(select_ids,1),
                    np.expand_dims([cls_ind] * len(select_ids),1)), 1)

            object_selected_dets.append(boxes_scores_rpn_id_label)

        object_selected_dets = np.concatenate(object_selected_dets)
        max_score_idx = np.argmax(object_selected_dets[:,4])
        object_selected_det = object_selected_dets[max_score_idx,:]
        
        
        
        try:
            selected_dets = np.concatenate((selected_dets, object_selected_det[None, :]))
            start_end_ids[int(object_selected_det[6]-1)] = [1,2]
        except Exception as e:
            print(e)
            import ipdb; ipdb.set_trace()
          
       
    return selected_dets, start_end_ids

In [6]:

io.mkdir_if_not_exists(data_const.proc_dir)

select_boxes_dir = data_const.proc_dir

# Print where the boxes are coming from and where the output is written
print(f'Boxes will be read from: {data_const.faster_rcnn_boxes}')
print(f'Boxes will be written to: {select_boxes_dir}')

print('Loading anno_list.json ...')
anno_list = io.load_json_object(data_const.anno_list_json)

print('Creating selected_coco_cls_dets.hdf5 file ...')
# hdf5_file = os.path.join(select_boxes_dir,'selected_coco_cls_dets.hdf5')
# f = h5py.File(hdf5_file,'w')


Boxes will be read from: /root/vs-gats-hhi/datasets/processed/hico/faster_rcnn_boxes
Boxes will be written to: /root/vs-gats-hhi/datasets/processed/hico
Loading anno_list.json ...
Creating selected_coco_cls_dets.hdf5 file ...


In [7]:

print('Selecting boxes ...')
for anno in tqdm(anno_list):
    global_id = anno['global_id']
    
    if global_id != 'ACCO_train2024_00000101':
        continue

    # # get more detection for evaluation
    # if 'test' in global_id:
    #     data_const.human_score_thresh = 0.1
    #     data_const.object_score_thresh = 0.1

    boxes_npy = os.path.join(
        data_const.faster_rcnn_boxes,
        f'{global_id}_boxes.npy')
    
    if not os.path.exists(boxes_npy):
        print("Skipped because boxes_npy does not exist:", boxes_npy)
        continue

    # print('boxes_npy:', boxes_npy)
    boxes = np.load(boxes_npy)
    
    scores_npy = os.path.join(
        data_const.faster_rcnn_boxes,
        f'{global_id}_scores.npy')
    scores = np.load(scores_npy)
    print()
    print("scores shape:", scores.shape, "--> (num. boxes, num. classes)")
    print()
    
    nms_keep_indices_json = os.path.join(
        data_const.faster_rcnn_boxes,
        f'{global_id}_nms_keep_indices.json')
    nms_keep_indices = io.load_json_object(nms_keep_indices_json)

    # import ipdb; ipdb.set_trace()
    # selected_dets, start_end_ids = select_dets(boxes, scores, nms_keep_indices, data_const)
    # selected_dets_supervisor, start_end_ids_supervisor = select_dets_supervisor(boxes,scores,nms_keep_indices,data_const)
    selected_dets, start_end_ids = select_dets_custom(boxes, scores, nms_keep_indices)
    
    # ================
    # DEBUGGING
    # ================
    # print("selected_dets with 'supervisor' class:")
    # print(selected_dets_supervisor[:, -1], "--> therre's index 8 for 'supervisor'")
    # print()
    # print("selected_dets (worker_idx --> supervisor_idx):")
    # print(selected_dets[:, -1], "--> if it has the same length with selected_dets_supervisor, then correct")
    print()
    print("selected_dets.shape:", selected_dets.shape)
    print("det_boxes:", selected_dets[:, :4])
    print("det_class:", selected_dets[:, -1])
    
    # f.create_group(global_id)
    # f[global_id].create_dataset('boxes_scores_rpn_ids',data=selected_dets)
    # f[global_id].create_dataset('start_end_ids',data=start_end_ids)

    # Test for 1 iter. for debugging
    break
# f.close()

Selecting boxes ...


  0%|          | 0/5557 [00:00<?, ?it/s]


scores shape: (200, 161) --> (num. boxes, num. classes)


selected_dets.shape: (48, 7)
det_boxes: [[2.23555328e+02 5.44837646e+02 3.03247223e+02 7.21757874e+02]
 [7.41688477e+02 5.04776428e+02 7.95287537e+02 6.89298340e+02]
 [1.33571021e+03 5.41778992e+02 1.37273010e+03 6.07477295e+02]
 [7.03833008e+02 5.21233276e+02 7.63379822e+02 6.95136841e+02]
 [1.09478235e+03 6.91292480e+02 1.15651477e+03 7.56093262e+02]
 [1.04811267e+03 5.57281555e+02 1.09219861e+03 6.61834900e+02]
 [1.48723828e+03 5.36459412e+02 1.54644946e+03 6.94093323e+02]
 [1.35830554e+03 7.91841675e+02 1.43103125e+03 8.60940918e+02]
 [3.58506348e+02 5.60481506e+02 4.29151611e+02 7.25079590e+02]
 [2.19846252e+02 5.39949585e+02 3.03822693e+02 7.39478882e+02]
 [8.23431030e+02 7.08194946e+02 8.75248291e+02 7.59444580e+02]
 [1.04919519e+03 5.58723389e+02 1.09158032e+03 6.57373535e+02]
 [7.01380920e+02 5.29238159e+02 7.64946655e+02 6.90476074e+02]
 [2.88945103e+01 5.57683411e+02 1.18429260e+02 6.80784058e+02]
 [1.78151758e+03 5.

In [8]:
selected_dets[:, -1]

array([ 1.,  2.,  2.,  2.,  2.,  2.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,
        4.,  5.,  1.,  2.,  3.,  7.,  8.,  6.,  7.,  8.,  8.,  8.,  8.,
        8.,  8., 10., 10., 10., 10., 10., 10., 10.,  9.,  9.,  9.,  9.,
        9.,  9.,  9.,  9.,  9.,  9.,  9.,  9.,  9.])